# CESM 예산 과제에 대한 추가 힌트 및 안내

이 노트북은 올버니 대학교(University at Albany)의 [Brian E. J. Rose](https://www.atmos.albany.edu/facstaff/brose/) 교수가 작성한 [기후 실험실 (The Climate Laboratory)](https://brian-rose.github.io/ClimateLaboratoryBook)의 일부입니다.

## 서문

과제에서는 CESM 제어 시뮬레이션으로부터 전지구 평균 에너지 수지의 다양한 항들을 계산하고 이를 관측 자료와 비교하도록 요구됩니다.

강의록에서는 데이터셋을 열고 시간 및 전지구 평균을 계산하는 방법을 살펴보았습니다.

여기서 주요 난제는 CESM 출력 데이터셋에 있는 진단 변수들(diagnostic fields)을 계산하도록 요구되는 양들과 어떻게 연결할지 이해하는 것입니다. 여기에서 몇 가지 추가적인 지침을 드리겠습니다.

## "상향 복사(upwelling)", "하향 복사(downwelling)", "순(net)"은 무엇을 의미하는가?

이 용어들을 이해하려면, 지구 표면과 대기권 최상단 사이를 이동하는 복사선(radiation)을 상상해 보세요. "하향 복사(Downwelling)"는 아래쪽으로(즉, 표면을 향해) 이동하는 것을 의미하며, "상향 복사(upwelling)"는 위쪽으로 이동하는 것을 의미합니다.

단파 복사(shortwave radiation, 태양 복사)는 태양으로부터 대기권 최상단에 도달하여 아래로 이동합니다. 이 복사선 중 일부는 위쪽으로 다시 반사됩니다. 따라서 우리가 노트에서 Q로 표기했던 양, 즉 지구에 입사하는 총 태양광(sunlight)은 대기권 최상단의 하향 단파 복사속(downwelling shortwave flux)입니다. 대기권 최상단의 상향 단파 복사속(upwelling shortwave flux)은 Q보다 작은 값이며 반사량(reflection)을 나타냅니다.

장파 복사(longwave radiation)의 경우에도 같은 개념입니다. 위쪽으로 이동하는 복사, 즉 "상향 복사(upwelling)"와 아래쪽으로 이동하는 복사, 즉 "하향 복사(downwelling)"가 있습니다. 차이점은 이 복사선이 지구와 대기 자체에서 생성된다는 것입니다.
표면에서의 상향 복사속(upwelling flux)은 표면 방출(emission)입니다. 표면에서의 하향 복사속(downwelling flux)은 (대기권 어딘가에서 발생하여) 표면에 흡수되는 복사입니다.

"순(net)" 복사속은 상향 복사와 하향 복사 간의 차이입니다. 부호 규약(sign conventions)에 유의해야 합니다! CESM(기후 시스템 모델) 데이터에서 "순(net)"은 장파 복사에 대해서는 순 상향(net upward)을 의미하지만, 단파 복사에 대해서는 순 하향(net downward)을 의미합니다.

대기권 최상단의 순 장파 복사속은 우리가 외향 장파 복사(Outgoing Longwave Radiation, OLR)라고 불렀던 것입니다.

대기권 최상단의 순 단파 복사속은 우리가 흡수 단파 복사(Absorbed Shortwave Radiation, ASR)라고 불렀던 것입니다.

## 강의 노트에서 발췌 -- CESM 명명 규칙

강의 노트에서 발췌:

모델 출력에는 복사 플럭스(radiative fluxes)에 대한 많은 진단 자료가 포함되어 있습니다. 적절한 출력 필드(output fields)를 찾는 데 도움이 될 만한 CESM 명명 규칙 몇 가지를 소개합니다:

- 이름이 `'F'`로 시작하는 모든 변수는 일종의 **에너지 플럭스**(energy flux)입니다.
- 대부분은 네 글자로 된 코드를 가지며, 예시는 `'FLNT'`입니다.
- `'FL'`은 **장파 플럭스**(longwave flux)를 의미합니다 (즉, 지구 복사).
- `'FS'`는 **단파 플럭스**(shortwave flux)를 의미합니다 (즉, 태양 복사).
- 세 번째 글자는 플럭스의 **방향**(direction)을 나타냅니다:
    - `'U'` = 위쪽
    - `'D'` = 아래쪽
    - `'N'` = 순(net)
- 네 번째 글자는 플럭스의 **위치**(location)를 나타냅니다:
    - `'T'` = 대기 상단(top of atmosphere)
    - `'S'` = 지표면(surface)
- 따라서 `'FLNT'`는 '대기 상단에서의 순 장파 플럭스'를 의미하며, 이는 외향 장파 복사(Outgoing Longwave Radiation, OLR)입니다.

이 모든 변수들은 240 x 96 x 144 크기를 가지며 -- 즉, 시뮬레이션의 매월에 대한 2차원 격자입니다.

이러한 명명 규칙의 한 가지 예외는, 강의 노트에서 보았듯이, 입사 태양 복사 (또는 일사량/Insolation)가 데이터셋에서는 **not** `FSDT` ("상단에서 아래로 향하는 단파 복사 플럭스")라고 불리지 않고 대신 `SOLIN`이라고 불린다는 점입니다.

## 데이터를 불러오고 간단히 살펴보기

In [1]:
import xarray as xr

cesm_data_path = "http://thredds.atmos.albany.edu:8080/thredds/dodsC/CESMA/"
# For better performance if you can access the roselab_rit filesystem (e.g. from JupyterHub)
# cesm_data_path = "/roselab_rit/cesm_archive/"
atm_control = xr.open_dataset(cesm_data_path + "cpl_1850_f19/concatenated/cpl_1850_f19.cam.h0.nc")

위에 제시된 명명 규칙에 따르면, 지표면에서의 하향 장파 복사(downwelling longwave radiation)는 `atm_control.FLDS`로 찾을 수 있습니다. 이는 "지표면에서 아래로 향하는 장파 플럭스(Flux of Longwave traveling Downward at the Surface)"를 나타내기 때문입니다.

In [2]:
atm_control.FLDS

<xarray.DataArray 'FLDS' (time: 240, lat: 96, lon: 144)> Size: 13MB
[3317760 values with dtype=float32]
Coordinates:
  * time     (time) object 2kB 0001-02-01 00:00:00 ... 0021-01-01 00:00:00
  * lat      (lat) float64 768B -90.0 -88.11 -86.21 -84.32 ... 86.21 88.11 90.0
  * lon      (lon) float64 1kB 0.0 2.5 5.0 7.5 10.0 ... 350.0 352.5 355.0 357.5
Attributes:
    Sampling_Sequence:  rad_lwsw
    units:              W/m2
    long_name:          Downwelling longwave flux at surface
    cell_methods:       time: mean

So that works!

Now, let's see if we can find the reflected shortwave radiation at the top of the atmosphere as `atm_control.FSUT`, since it is "Flux of Shortwave traveling Upward at the Top".

In [3]:
atm_control.FSUT

AttributeError: 'Dataset' object has no attribute 'FSUT'

이 필드는 데이터셋에 없는 것 같습니다!

## 데이터셋에 없는 필드 계산하기

우리가 필요로 하는 일부 필드는 데이터셋에 직접 저장되어 있지 않습니다.

하지만 괜찮습니다. 왜냐하면 우리가 가지고 있는 필드들을 사용하여 간단한 산술 계산으로 이들을 계산할 수 있는 충분한 정보가 있기 때문입니다.

누락된 `FSUT` 필드의 경우, 핵심은 우리가 "순(Net)" 필드인 `FSNT`와 하향 복사(downwelling) 필드인 `SOLIN`을 가지고 있다는 점입니다.

순 복사량(Net Flux)은 차이값이므로:

`FSNT = SOLIN - FSUT`

(즉, 들어오는 것과 나가는 것의 차이!)

그러면 이 식을 재배열하여 우리가 모르는 필드를 구할 수 있습니다:

`FSUT = SOLIN - FSNT`

우리는 xarray 데이터셋을 사용하여 이러한 종류의 산술 연산을 수행할 수 있습니다:

In [4]:
#  Take the difference between the downwelling shortwave flux at the net shortwave flux at the top of atmosphere
#  The result is the upwelling flux, i.e. the reflected shortwave flux
#   Store this difference in a new variable called FSUT
FSUT = atm_control.SOLIN - atm_control.FSNT

### 몇 가지 추가 파생 변수(derived field) 예시

과제에서 여러분은 장파(longwave) 형태의 "지표면으로부터의 상향 방출(Upward emission from the surface)"을 계산하게 됩니다. 이 값은 `FLUS`입니다.

In [5]:
atm_control.FLUS

AttributeError: 'Dataset' object has no attribute 'FLUS'

위와 동일한 문제이며, 해결 방법도 같습니다.

우리에게는 하향 복사 플럭스 `FLDS` (아래 방향이 양수)와 순 복사 플럭스 `FLNS` (위 방향이 양수)가 주어져 있습니다. 순 복사 플럭스는 다음과 같이 정의됩니다.

`FLNS = FLUS - FLDS`

위 식을 재배열하여 상향 복사 플럭스를 얻을 수 있습니다:

`FLUS = FLNS + FLDS`

In [6]:
FLUS = atm_control.FLDS + atm_control.FLNS

우리 모두 이러한 부호 규약(sign conventions)에 대해 헷갈려 합니다! 상하 방향으로 이동하는 플럭스(fluxes)를 직접 그려보는 것이 도움이 됩니다.

우리의 결과가 물리적으로 타당한지 또한 검증해야 합니다. 만약 `FLUS` 장의 전 지구적, 시간 평균을 취한다면, 이는 큰 양수 값이어야 합니다!

이 값이 396 W/m2인 관측값과 비교하고 있음을 기억하세요.

In [7]:
#  don't forget to take the area-weighted average!
weight_factor2 = atm_control.gw / atm_control.gw.mean(dim='lat')
#  I'm going to calculate the average and print it out, rounded to the second decimal place
myvalue = (FLUS*weight_factor2).mean(dim=('time','lat','lon')).values  
# note that adding .values at the end of the expression gives me just the number without the metadata
print('Upwelling longwave radiation at the surface: {:.2f} W/m2'.format(myvalue))   
# This is an example of a formatted print statement in Python

Upwelling longwave radiation at the surface: 395.45 W/m2


이는 관측값과 겉보기에는 매우 유사하며, 우리가 계산을 올바르게 수행했음을 나타내는 좋은 지표입니다.

## 출처

이 노트북은 University at Albany의 [Brian E. J. Rose](https://www.atmos.albany.edu/facstaff/brose/)가 개발하고 유지 관리하는 오픈 소스 교재인 [The Climate Laboratory](https://brian-rose.github.io/ClimateLaboratoryBook)의 일부입니다.

이 자료는 [크리에이티브 커먼즈 저작자표시 4.0 국제 (CC BY 4.0)](https://creativecommons.org/licenses/by/4.0/) 라이선스에 따라 무료로 자유롭게 이용할 수 있습니다.

이 자료와 [climlab 소프트웨어](https://github.com/climlab/climlab)의 개발은 Brian Rose에게 수여된 AGS-1455071 과제를 통해 미국 국립과학재단(National Science Foundation)의 부분적인 지원을 받았습니다. 여기에 표현된 모든 의견, 연구 결과, 결론 또는 권고사항은 저의 것이며, 반드시 미국 국립과학재단(National Science Foundation)의 견해를 반영하는 것은 아닙니다.